In [1]:
# Imports

import pandas as pd
import numpy as np

from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    average_precision_score,
    f1_score,
    recall_score,
    precision_score
)

from catboost import CatBoostClassifier

import warnings
warnings.filterwarnings("ignore")

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)


In [2]:
# Metrics helper for imbalanced classification (PR-AUC primary, plus F1/recall/precision)
from datetime import datetime

def compute_metrics(y_true, y_proba, threshold=0.5):
    """
    Compute core metrics for imbalanced classification.
    Primary metric: PR-AUC
    Secondary: F1, Recall, Precision
    """
    
    # Basic sanity checks
    assert len(y_true) == len(y_proba), "Length mismatch between labels and probabilities."
    assert np.all((y_proba >= 0) & (y_proba <= 1)), "Probabilities must be between 0 and 1."
    
    # Binary predictions from threshold
    y_pred = (y_proba >= threshold).astype(int)
    
    metrics_dict = {
        "PR_AUC": average_precision_score(y_true, y_proba),
        "F1": f1_score(y_true, y_pred),
        "Recall": recall_score(y_true, y_pred),
        "Precision": precision_score(y_true, y_pred)
    }
    
    return metrics_dict


def evaluate_model(model, X, y, dataset_name="Validation"):
    """
    Evaluate model on a dataset and return metrics.
    """
    
    y_proba = model.predict_proba(X)[:, 1]
    results = compute_metrics(y, y_proba)
    
    print(f"\n--- {dataset_name} Results ---")
    for k, v in results.items():
        print(f"{k:10s}: {v:.4f}")
    
    return results


def save_results_to_excel(results_list, filename="results.xlsx"):
    """
    Save experiment results to Excel.
    """
    
    df = pd.DataFrame(results_list)
    df["timestamp"] = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    
    df.to_excel(filename, index=False)
    
    print(f"\nResults saved to {filename}")
    return df

In [3]:
# Smoke-test compute_metrics
y_true = np.array([0,0,0,1,1])
y_proba = np.array([0.1,0.2,0.3,0.7,0.9])

print(compute_metrics(y_true, y_proba))

{'PR_AUC': 1.0, 'F1': 1.0, 'Recall': 1.0, 'Precision': 1.0}


In [4]:
# Input paths
paths = {
    'SPECTER': r"data/specter_embeddings.csv", 
        'BERT': r"data/bert_embeddings.csv",        
        'SciBERT': r"data/scibert_embeddings.csv",  
        'target': r"data/targets.csv",                
    "Dataset":r"data/combined_acl_arxiv_with_citations_final.csv",
    "split":r"train_val_test_split.json"
}

In [5]:
# Load dataset, SPECTER embeddings and targets
data = pd.read_csv(paths["Dataset"])
specter = pd.read_csv(paths["SPECTER"])
targets = pd.read_csv(paths["target"])

print("Dataset shape:", data.shape)
print("Specter shape:", specter.shape)
print("Targets shape:", targets.shape)

Dataset shape: (131772, 9)
Specter shape: (131772, 770)
Targets shape: (131772, 2)


In [20]:
# Preview targets
targets.head()

,project_id,is_highly_cited
0,0,0
1,2,0
2,3,0
3,5,0
4,6,0


In [6]:
# Confirm project_id is unique in each table before merging
print(data["project_id"].is_unique,
specter["project_id"].is_unique,
targets["project_id"].is_unique)

True True True


In [7]:
# Merge dataset + SPECTER + targets on project_id
df = data.merge(specter, on="project_id", how="inner")
df = df.merge(targets, on="project_id", how="inner")

print("Merged shape:", df.shape)

Merged shape: (131772, 779)


In [8]:
# Stylometric feature extractor (word/sentence stats, lexical diversity)

import re
import textstat

def compute_stylometric_features(text):
    
    if pd.isna(text):
        text = ""
        
    words = re.findall(r'\b\w+\b', text)
    sentences = re.split(r'[.!?\n]+', text)    
    word_count = len(words)
    sentence_count = len([s for s in sentences if len(s.strip()) > 0])
    
    avg_word_length = np.mean([len(w) for w in words]) if word_count > 0 else 0
    avg_sentence_length = word_count / sentence_count if sentence_count > 0 else 0
    
    lexical_diversity = len(set(words)) / word_count if word_count > 0 else 0
    
    complex_words = [w for w in words if len(w) >= 7]
    complex_word_ratio = len(complex_words) / word_count if word_count > 0 else 0
    
    # Readability metrics
    # flesch_reading_ease = textstat.flesch_reading_ease(text) if word_count > 0 else 0
    # flesch_kincaid_grade = textstat.flesch_kincaid_grade(text) if word_count > 0 else 0
    
    return pd.Series({
        "word_count": word_count,
        "sentence_count": sentence_count,
        "avg_word_length": avg_word_length,
        "avg_sentence_length": avg_sentence_length,
        "lexical_diversity": lexical_diversity,
        "complex_word_ratio": complex_word_ratio,
        # "flesch_reading_ease": flesch_reading_ease,
        # "flesch_kincaid_grade": flesch_kincaid_grade
    })

In [9]:
# Smoke-test on sample text
sample_text = "This is a simple test. It has two sentences."

print(compute_stylometric_features(sample_text))

word_count             9.000000
sentence_count         2.000000
avg_word_length        3.777778
avg_sentence_length    4.500000
lexical_diversity      1.000000
complex_word_ratio     0.111111
dtype: float64


In [10]:
# Smoke-test on empty text
print(compute_stylometric_features(""))

word_count             0
sentence_count         0
avg_word_length        0
avg_sentence_length    0
lexical_diversity      0
complex_word_ratio     0
dtype: int64


In [11]:
# Build combined title+abstract text field
df["combined_text"] = (
    df["title"].fillna("") + " " + df["abstract"].fillna("")
)

In [12]:
# Compute stylometric features
stylometric_df = df["combined_text"].apply(compute_stylometric_features)

In [13]:
# Summary of stylometric features
stylometric_df.describe()

,word_count,sentence_count,avg_word_length,avg_sentence_length,lexical_diversity,complex_word_ratio
count,131772.000000,131772.000000,131772.000000,131772.000000,131772.000000,131772.000000
mean,159.807683,14.732212,5.582162,15.632253,0.678372,0.376624
std,57.496980,9.853838,0.412429,9.583078,0.072513,0.060867
min,9.000000,1.000000,1.112281,1.576923,0.078363,0.000000
25%,120.000000,6.000000,5.328671,8.000000,0.631579,0.336898
50%,157.000000,12.000000,5.579545,9.600000,0.676471,0.376238
75%,194.000000,23.000000,5.839506,22.500000,0.723881,0.416185
max,1462.000000,264.000000,8.750000,261.000000,1.000000,0.680365


In [14]:
# Attach stylometric feature columns
stylometric_cols = [
    "word_count",
    "sentence_count",
    "avg_word_length",
    "avg_sentence_length",
    "lexical_diversity",
    "complex_word_ratio"
]

df = pd.concat([df, stylometric_df[stylometric_cols]], axis=1)

print("Updated df shape:", df.shape)

Updated df shape: (131772, 786)


In [15]:
# Apply the fixed train/val/test split
import json

with open(paths["split"], "r") as f:
    split_dict = json.load(f)

train_ids = set(split_dict["train"])
val_ids   = set(split_dict["val"])
test_ids  = set(split_dict["test"])

train_df = df[df["project_id"].isin(train_ids)].copy()
val_df   = df[df["project_id"].isin(val_ids)].copy()
test_df  = df[df["project_id"].isin(test_ids)].copy()

print("Train:", train_df.shape)
print("Val:", val_df.shape)
print("Test:", test_df.shape)

assert len(train_df) == len(train_ids)
assert len(val_df) == len(val_ids)
assert len(test_df) == len(test_ids)

print("Split integrity passed.")

Train: (104685, 786)
Val: (20129, 786)
Test: (6958, 786)
Split integrity passed.


In [16]:
# Column list
df.columns

Index(['project_id', 'original_id', 'abstract', 'citation_count',
       'publication_year', 'title', 'authors_str', 'source',
       'original_id_clean_x', 'original_id_clean_y',
       ...
       'specter_embedding_766', 'specter_embedding_767', 'is_highly_cited',
       'combined_text', 'word_count', 'sentence_count', 'avg_word_length',
       'avg_sentence_length', 'lexical_diversity', 'complex_word_ratio'],
      dtype='object', length=786)

In [17]:
# Collect the SPECTER feature columns
specter_cols = [c for c in df.columns if c.startswith("specter_")]

print("Specter feature count:", len(specter_cols))

Specter feature count: 768


In [21]:
# Feature set: SPECTER + stylometric (+ target)
TARGET_COL = "is_highly_cited"

MODEL_COLS = specter_cols + stylometric_cols + [TARGET_COL]

In [22]:
# Build X/y matrices for train/val/test
X_train = train_df[MODEL_COLS].drop(columns=[TARGET_COL])
y_train = train_df[TARGET_COL]

X_val = val_df[MODEL_COLS].drop(columns=[TARGET_COL])
y_val = val_df[TARGET_COL]

X_test = test_df[MODEL_COLS].drop(columns=[TARGET_COL])
y_test = test_df[TARGET_COL]

print("X_train shape:", X_train.shape)

X_train shape: (104685, 774)


In [23]:
# Class balance of the training set
y_train.value_counts()

is_highly_cited
0    83360
1    21325
Name: count, dtype: int64

In [24]:
# Class weights to offset the imbalance
from collections import Counter

counter = Counter(y_train)
total = len(y_train)

class_weights = {
    0: total / (2 * counter[0]),
    1: total / (2 * counter[1])
}

print("Class weights:", class_weights)

Class weights: {0: 0.6279090690978887, 1: 2.4545134818288394}


In [25]:
# Train a baseline CatBoost (fixed hyperparameters)
from catboost import CatBoostClassifier

model = CatBoostClassifier(
    iterations=2000,
    learning_rate=0.05,
    depth=6,
    eval_metric="AUC",          # internal metric
    loss_function="Logloss",
    class_weights=class_weights,
    random_seed=42,
    early_stopping_rounds=100,
    verbose=200
)

model.fit(
    X_train, y_train,
    eval_set=(X_val, y_val),
    use_best_model=True
)

0:	test: 0.6032274	best: 0.6032274 (0)	total: 615ms	remaining: 20m 30s
200:	test: 0.6895992	best: 0.6895992 (200)	total: 45.5s	remaining: 6m 47s
400:	test: 0.7016400	best: 0.7017593 (397)	total: 1m 28s	remaining: 5m 54s
600:	test: 0.7055896	best: 0.7055896 (600)	total: 2m 12s	remaining: 5m 7s
800:	test: 0.7078127	best: 0.7078127 (800)	total: 2m 56s	remaining: 4m 24s
1000:	test: 0.7082656	best: 0.7087475 (942)	total: 3m 36s	remaining: 3m 36s
Stopped by overfitting detector  (100 iterations wait)

bestTest = 0.7087474977
bestIteration = 942

Shrink model to first 943 iterations.


In [26]:
# Evaluate the baseline model on val and test
val_results = evaluate_model(model, X_val, y_val, dataset_name="Validation")
test_results = evaluate_model(model, X_test, y_test, dataset_name="Test")


--- Validation Results ---
PR_AUC    : 0.3698
F1        : 0.3912
Recall    : 0.9065
Precision : 0.2494

--- Test Results ---
PR_AUC    : 0.3996
F1        : 0.4598
Recall    : 0.8857
Precision : 0.3105


In [27]:
# Threshold search maximizing validation F1
def find_best_threshold(y_true, y_proba):
    thresholds = np.linspace(0.05, 0.95, 100)
    
    best_f1 = 0
    best_thresh = 0.5
    
    for t in thresholds:
        y_pred = (y_proba >= t).astype(int)
        f1 = f1_score(y_true, y_pred)
        
        if f1 > best_f1:
            best_f1 = f1
            best_thresh = t
    
    return best_thresh, best_f1

In [28]:
# Pick the best threshold on validation probabilities
val_proba = model.predict_proba(X_val)[:, 1]

best_thresh, best_f1 = find_best_threshold(y_val, val_proba)

print("Best threshold:", best_thresh)
print("Best validation F1:", best_f1)

Best threshold: 0.6681818181818182
Best validation F1: 0.42984213478644123


In [29]:
# Metrics at the tuned threshold (val + test)
val_results = compute_metrics(y_val, val_proba, threshold=best_thresh)
print("Val:", val_results)

test_proba = model.predict_proba(X_test)[:, 1]
test_results = compute_metrics(y_test, test_proba, threshold=best_thresh)
print("Test:", test_results)

Val: {'PR_AUC': 0.36982628903681874, 'F1': 0.42984213478644123, 'Recall': 0.6597785977859778, 'Precision': 0.3187544568576183}
Test: {'PR_AUC': 0.3996194374077984, 'F1': 0.4586833365891776, 'Recall': 0.615626638699528, 'Precision': 0.3655043586550436}


In [30]:
# train_and_evaluate: full fit + threshold tune + save routine
import numpy as np
import json
import joblib
from catboost import CatBoostClassifier
from sklearn.metrics import (
    precision_recall_curve,
    average_precision_score,
    f1_score,
    precision_score,
    recall_score
)

def train_and_evaluate(
    X_train, y_train,
    X_val, y_val,
    X_test, y_test,
    model_name,
    save_dir="results"
):
    
    print(f"\n===== Training {model_name} =====")

    model = CatBoostClassifier(
        iterations=2000,
        learning_rate=0.05,
        depth=6,
        loss_function="Logloss",
        eval_metric="AUC",
        random_seed=42,
        early_stopping_rounds=100,
        verbose=200
    )

    model.fit(
        X_train, y_train,
        eval_set=(X_val, y_val),
        use_best_model=True
    )

    # --- Probabilities ---
    val_probs = model.predict_proba(X_val)[:, 1]
    test_probs = model.predict_proba(X_test)[:, 1]

    # --- PR-AUC ---
    val_pr_auc = average_precision_score(y_val, val_probs)
    test_pr_auc = average_precision_score(y_test, test_probs)

    # --- Threshold tuning (VALIDATION ONLY) ---
    thresholds = np.linspace(0.05, 0.95, 200)

    best_thr = 0.5
    best_f1 = 0

    for t in thresholds:
        preds = (val_probs >= t).astype(int)
        f1 = f1_score(y_val, preds)
        if f1 > best_f1:
            best_f1 = f1
            best_thr = t

    print(f"\nBest threshold (val): {best_thr}")
    print(f"Best validation F1: {best_f1}")

    # --- Final metrics using best threshold ---
    val_preds = (val_probs >= best_thr).astype(int)
    test_preds = (test_probs >= best_thr).astype(int)

    val_metrics = {
        "PR_AUC": float(val_pr_auc),
        "F1": float(f1_score(y_val, val_preds)),
        "Recall": float(recall_score(y_val, val_preds)),
        "Precision": float(precision_score(y_val, val_preds)),
    }

    test_metrics = {
        "PR_AUC": float(test_pr_auc),
        "F1": float(f1_score(y_test, test_preds)),
        "Recall": float(recall_score(y_test, test_preds)),
        "Precision": float(precision_score(y_test, test_preds)),
    }

    results = {
        "model_name": model_name,
        "best_iteration": model.get_best_iteration(),
        "best_threshold": float(best_thr),
        "validation": val_metrics,
        "test": test_metrics
    }

    # --- Save model ---
    joblib.dump(model, f"{save_dir}/{model_name}_model.pkl")

    # --- Save results ---
    with open(f"{save_dir}/{model_name}_metrics.json", "w") as f:
        json.dump(results, f, indent=4)

    print("\n--- Validation ---")
    print(val_metrics)
    print("\n--- Test ---")
    print(test_metrics)

    return model, results

In [32]:
# Train and evaluate the SPECTER model
specter_features = specter_cols + stylometric_cols

model_specter, results_specter = train_and_evaluate(
    X_train[specter_features], y_train,
    X_val[specter_features], y_val,
    X_test[specter_features], y_test,
    model_name="specter"
)


===== Training specter =====
0:	test: 0.5979802	best: 0.5979802 (0)	total: 233ms	remaining: 7m 45s
200:	test: 0.6981075	best: 0.6981075 (200)	total: 51.3s	remaining: 7m 38s
400:	test: 0.7062837	best: 0.7063375 (399)	total: 1m 45s	remaining: 6m 59s
600:	test: 0.7101754	best: 0.7101754 (600)	total: 2m 38s	remaining: 6m 8s
800:	test: 0.7120034	best: 0.7121797 (793)	total: 3m 28s	remaining: 5m 11s
Stopped by overfitting detector  (100 iterations wait)

bestTest = 0.7121797098
bestIteration = 793

Shrink model to first 794 iterations.

Best threshold (val): 0.3801507537688442
Best validation F1: 0.432770588739395

--- Validation ---
{'PR_AUC': 0.37499189446644965, 'F1': 0.432770588739395, 'Recall': 0.6211562115621156, 'Precision': 0.3320620725933719}

--- Test ---
{'PR_AUC': 0.40362144683202617, 'F1': 0.4567901234567901, 'Recall': 0.5820660723649712, 'Precision': 0.37588892651540806}


In [34]:
# Merge in SciBERT embeddings
sci_df = pd.read_csv(paths["SciBERT"])
df = df.merge(sci_df, on="project_id", how="inner")

In [36]:
# Collect the SciBERT feature columns
scibert_cols = [c for c in df.columns if c.startswith("scibert_")]
print("SciBERT feature count:", len(scibert_cols))
scibert_features = scibert_cols + stylometric_cols
print(scibert_features)

SciBERT feature count: 768
['scibert_embedding_0', 'scibert_embedding_1', 'scibert_embedding_2', 'scibert_embedding_3', 'scibert_embedding_4', 'scibert_embedding_5', 'scibert_embedding_6', 'scibert_embedding_7', 'scibert_embedding_8', 'scibert_embedding_9', 'scibert_embedding_10', 'scibert_embedding_11', 'scibert_embedding_12', 'scibert_embedding_13', 'scibert_embedding_14', 'scibert_embedding_15', 'scibert_embedding_16', 'scibert_embedding_17', 'scibert_embedding_18', 'scibert_embedding_19', 'scibert_embedding_20', 'scibert_embedding_21', 'scibert_embedding_22', 'scibert_embedding_23', 'scibert_embedding_24', 'scibert_embedding_25', 'scibert_embedding_26', 'scibert_embedding_27', 'scibert_embedding_28', 'scibert_embedding_29', 'scibert_embedding_30', 'scibert_embedding_31', 'scibert_embedding_32', 'scibert_embedding_33', 'scibert_embedding_34', 'scibert_embedding_35', 'scibert_embedding_36', 'scibert_embedding_37', 'scibert_embedding_38', 'scibert_embedding_39', 'scibert_embedding_40'

In [38]:
# Feature set: SciBERT + stylometric (+ target)
MODEL_COLS = scibert_features + [TARGET_COL]

In [40]:
# Re-apply the split for the SciBERT features
import json

with open(paths["split"], "r") as f:
    split_dict = json.load(f)

train_ids = set(split_dict["train"])
val_ids   = set(split_dict["val"])
test_ids  = set(split_dict["test"])

train_df = df[df["project_id"].isin(train_ids)].copy()
val_df   = df[df["project_id"].isin(val_ids)].copy()
test_df  = df[df["project_id"].isin(test_ids)].copy()

print("Train:", train_df.shape)
print("Val:", val_df.shape)
print("Test:", test_df.shape)

assert len(train_df) == len(train_ids)
assert len(val_df) == len(val_ids)
assert len(test_df) == len(test_ids)

print("Split integrity passed.")

Train: (104685, 1555)
Val: (20129, 1555)
Test: (6958, 1555)
Split integrity passed.


In [41]:
# Build X/y for the SciBERT features
X_train = train_df[MODEL_COLS].drop(columns=[TARGET_COL])
y_train = train_df[TARGET_COL]

X_val = val_df[MODEL_COLS].drop(columns=[TARGET_COL])
y_val = val_df[TARGET_COL]

X_test = test_df[MODEL_COLS].drop(columns=[TARGET_COL])
y_test = test_df[TARGET_COL]

print("X_train shape:", X_train.shape)

X_train shape: (104685, 774)


In [42]:
# Train and evaluate the SciBERT model
model_scibert, results_scibert = train_and_evaluate(
    X_train[scibert_features], y_train,
    X_val[scibert_features], y_val,
    X_test[scibert_features], y_test,
    model_name="scibert"
)


===== Training scibert =====
0:	test: 0.5624712	best: 0.5624712 (0)	total: 224ms	remaining: 7m 26s
200:	test: 0.6841268	best: 0.6841268 (200)	total: 45.7s	remaining: 6m 49s
400:	test: 0.6983504	best: 0.6983504 (400)	total: 1m 25s	remaining: 5m 42s
600:	test: 0.7034509	best: 0.7034858 (599)	total: 2m 4s	remaining: 4m 48s
800:	test: 0.7058926	best: 0.7059885 (795)	total: 2m 41s	remaining: 4m 2s
1000:	test: 0.7079827	best: 0.7080658 (991)	total: 3m 26s	remaining: 3m 25s
1200:	test: 0.7091852	best: 0.7091852 (1200)	total: 4m 9s	remaining: 2m 45s
1400:	test: 0.7109311	best: 0.7109311 (1400)	total: 4m 47s	remaining: 2m 3s
1600:	test: 0.7119224	best: 0.7119224 (1600)	total: 5m 27s	remaining: 1m 21s
1800:	test: 0.7128680	best: 0.7131637 (1758)	total: 6m 7s	remaining: 40.6s
Stopped by overfitting detector  (100 iterations wait)

bestTest = 0.7132555418
bestIteration = 1881

Shrink model to first 1882 iterations.

Best threshold (val): 0.3304020100502512
Best validation F1: 0.43937439535633666


In [46]:
# Drop duplicate id columns left by the merges
df.drop(columns=["original_id_clean_x","original_id_clean_y"], inplace=True)
print(df.columns)


Index(['project_id', 'original_id', 'abstract', 'citation_count',
       'publication_year', 'title', 'authors_str', 'source',
       'specter_embedding_0', 'specter_embedding_1',
       ...
       'scibert_embedding_758', 'scibert_embedding_759',
       'scibert_embedding_760', 'scibert_embedding_761',
       'scibert_embedding_762', 'scibert_embedding_763',
       'scibert_embedding_764', 'scibert_embedding_765',
       'scibert_embedding_766', 'scibert_embedding_767'],
      dtype='object', length=1553)


In [47]:
# Merge in BERT embeddings
bert_df = pd.read_csv(paths["BERT"])
df = df.merge(bert_df, on="project_id", how="inner")

In [48]:
# Collect the BERT feature columns
bert_cols = [c for c in df.columns if c.startswith("bert_")]
print("BERT feature count:", len(bert_cols))
bert_features = bert_cols + stylometric_cols
print(bert_features)

BERT feature count: 768
['bert_embedding_0', 'bert_embedding_1', 'bert_embedding_2', 'bert_embedding_3', 'bert_embedding_4', 'bert_embedding_5', 'bert_embedding_6', 'bert_embedding_7', 'bert_embedding_8', 'bert_embedding_9', 'bert_embedding_10', 'bert_embedding_11', 'bert_embedding_12', 'bert_embedding_13', 'bert_embedding_14', 'bert_embedding_15', 'bert_embedding_16', 'bert_embedding_17', 'bert_embedding_18', 'bert_embedding_19', 'bert_embedding_20', 'bert_embedding_21', 'bert_embedding_22', 'bert_embedding_23', 'bert_embedding_24', 'bert_embedding_25', 'bert_embedding_26', 'bert_embedding_27', 'bert_embedding_28', 'bert_embedding_29', 'bert_embedding_30', 'bert_embedding_31', 'bert_embedding_32', 'bert_embedding_33', 'bert_embedding_34', 'bert_embedding_35', 'bert_embedding_36', 'bert_embedding_37', 'bert_embedding_38', 'bert_embedding_39', 'bert_embedding_40', 'bert_embedding_41', 'bert_embedding_42', 'bert_embedding_43', 'bert_embedding_44', 'bert_embedding_45', 'bert_embedding_46'

In [49]:
# Feature set: BERT + stylometric (+ target)
MODEL_COLS = bert_features + [TARGET_COL]

In [51]:
# Drop the SciBERT/SPECTER columns to free memory
df.drop(columns=scibert_cols+specter_cols, inplace=True)

In [52]:
# Re-apply the split for the BERT features
import json

with open(paths["split"], "r") as f:
    split_dict = json.load(f)

train_ids = set(split_dict["train"])
val_ids   = set(split_dict["val"])
test_ids  = set(split_dict["test"])

train_df = df[df["project_id"].isin(train_ids)].copy()
val_df   = df[df["project_id"].isin(val_ids)].copy()
test_df  = df[df["project_id"].isin(test_ids)].copy()

print("Train:", train_df.shape)
print("Val:", val_df.shape)
print("Test:", test_df.shape)

assert len(train_df) == len(train_ids)
assert len(val_df) == len(val_ids)
assert len(test_df) == len(test_ids)

print("Split integrity passed.")

Train: (104685, 786)
Val: (20129, 786)
Test: (6958, 786)
Split integrity passed.


In [53]:
# Build X/y for the BERT features
X_train = train_df[MODEL_COLS].drop(columns=[TARGET_COL])
y_train = train_df[TARGET_COL]

X_val = val_df[MODEL_COLS].drop(columns=[TARGET_COL])
y_val = val_df[TARGET_COL]

X_test = test_df[MODEL_COLS].drop(columns=[TARGET_COL])
y_test = test_df[TARGET_COL]

print("X_train shape:", X_train.shape)

X_train shape: (104685, 774)


In [54]:
# Train and evaluate the BERT model
bert_features = bert_cols + stylometric_cols

model_bert, results_bert = train_and_evaluate(
    X_train[bert_features], y_train,
    X_val[bert_features], y_val,
    X_test[bert_features], y_test,
    model_name="bert"
)


===== Training bert =====
0:	test: 0.5728095	best: 0.5728095 (0)	total: 358ms	remaining: 11m 55s
200:	test: 0.6779512	best: 0.6780357 (199)	total: 1m 12s	remaining: 10m 49s
400:	test: 0.6900131	best: 0.6901401 (395)	total: 2m 15s	remaining: 9m 1s
600:	test: 0.6957763	best: 0.6957860 (599)	total: 3m 21s	remaining: 7m 50s
800:	test: 0.6986265	best: 0.6986265 (800)	total: 4m 27s	remaining: 6m 40s
1000:	test: 0.6995511	best: 0.6996599 (995)	total: 5m 28s	remaining: 5m 27s
1200:	test: 0.7004385	best: 0.7006014 (1156)	total: 6m 29s	remaining: 4m 19s
1400:	test: 0.7014995	best: 0.7014995 (1400)	total: 7m 28s	remaining: 3m 11s
1600:	test: 0.7019501	best: 0.7019607 (1598)	total: 8m 25s	remaining: 2m 5s
Stopped by overfitting detector  (100 iterations wait)

bestTest = 0.7019606996
bestIteration = 1598

Shrink model to first 1599 iterations.

Best threshold (val): 0.3394472361809045
Best validation F1: 0.4251383399209486

--- Validation ---
{'PR_AUC': 0.3738644868409333, 'F1': 0.425138339920948